In [2]:
from reservoirgrid.models import Reservoir
from reservoirgrid.helpers import utils, viz, chaos_utils

import numpy as np
from dysts import flows

c:\Users\jaych\ReservoirGrid\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
lorenzDriven = flows.Lorenz()
lorenzDriven.sigma = 10.0
lorenzDriven.beta = 8/3
lorenzDriven.rho = 210.0
DrivenLorenz = lorenzDriven.make_trajectory(10000, method='RK45', standardize=True, dt=0.01)

Lorenz = flows.Lorenz().make_trajectory(10000, method='RK45', standardize=True, dt=0.01)

In [31]:
#viz.plot_components(DrivenLorenz, title="Lorenz Attractor with rho=210.0")
viz.compare_plot([DrivenLorenz, Lorenz], labels=["Driven Lorenz", "Standard Lorenz"], title="Comparing Driven vs Standard Lorenz Attractor")

In [34]:
import numpy as np
from scipy.integrate import solve_ivp

# 1. Define the Lorenz System Equations and its analytic Jacobian Matrix
def lorenz_dynamics(t, state, sigma, beta, rho):
    x, y, z = state
    return [
        sigma * (y - x),
        x * (rho - z) - y,
        x * y - beta * z
    ]

def lorenz_jacobian(state, sigma, beta, rho):
    x, y, z = state
    return np.array([
        [-sigma, sigma, 0],
        [rho - z, -1, -x],
        [y, x, -beta]
    ])

# 2. Combined solver step for processing trajectory & perturbation frame together
def system_ode_wrapper(t, y_combined, sigma, beta, rho):
    # Split the flat array back into the 3D position vector and 3x3 perturbation frame
    state = y_combined[:3]
    frame = y_combined[3:].reshape((3, 3))
    
    # Calculate derivatives
    d_state = lorenz_dynamics(t, state, sigma, beta, rho)
    jac = lorenz_jacobian(state, sigma, beta, rho)
    d_frame = jac @ frame  # Linearized growth equation
    
    return np.concatenate([d_state, d_frame.flatten()])

# 3. Parameter setup (Standard Lorenz configuration)
sigma, beta, rho = 10.0, 8/3, 280.0
init_state = [1.0, 1.0, 1.0]
init_frame = np.eye(3) # Perfect orthogonal start sphere
y_init = np.concatenate([init_state, init_frame.flatten()])

# 4. Iterative Loop using Benettin's Orthonormalization Method
t_start = 0.0
dt = 0.01
total_steps = 5000
lyapunov_sums = np.zeros(3)

current_y = y_init

# Warm-up phase: let the initial transient settle onto the attractor
sol_warmup = solve_ivp(lorenz_dynamics, [0, 20], init_state, args=(sigma, beta, rho), rtol=1e-8, atol=1e-10)
current_y[:3] = sol_warmup.y[:, -1]

# Main computation loop
for step in range(total_steps):
    t_end = t_start + dt
    
    # Advance both the position and perturbation space forward by dt
    sol = solve_ivp(system_ode_wrapper, [t_start, t_end], current_y, args=(sigma, beta, rho), rtol=1e-8, atol=1e-10)
    current_y = sol.y[:, -1]
    
    # Extract the stretched perturbation frame
    stretched_frame = current_y[3:].reshape((3, 3))
    
    # QR Decomposition: Q is orthonormalized frame, R is the upper triangular matrix
    Q, R = np.linalg.qr(stretched_frame)
    
    # Log the stretching scale factors (absolute value of the diagonal elements of R)
    diag_R = np.abs(np.diag(R))
    lyapunov_sums += np.log(diag_R)
    
    # Reset current_y with the re-orthogonalized frame to keep calculations stable
    current_y[3:] = Q.flatten()
    t_start = t_end

# 5. Divide the logged sum by the complete elapsed time duration
lyapunov_exponents = lyapunov_sums / (total_steps * dt)

print("--- Final Computed Lyapunov Spectrum ---")
print(f"Positive Component (λ1 - Stretching Rate):  {lyapunov_exponents[0]:.4f}")
print(f"Zero Component     (λ2 - Flow Direction):   {lyapunov_exponents[1]:.4f}")
print(f"Negative Component (λ3 - Compression Rate): {lyapunov_exponents[2]:.4f}")


--- Final Computed Lyapunov Spectrum ---
Positive Component (λ1 - Stretching Rate):  -0.0080
Zero Component     (λ2 - Flow Direction):   -1.9372
Negative Component (λ3 - Compression Rate): -11.7214
